# Nokken systeem 28

Deze notebook maakt dezelfde nokbeweging op twee manieren:

1. **5de-graads bewegingswet**
2. **Cycloidale bewegingswet**

Gevraagde beweging:

- $20^\circ \rightarrow 90^\circ$: $+25\,\mathrm{mm}$
- $90^\circ \rightarrow 155^\circ$: $-15\,\mathrm{mm}$
- $155^\circ \rightarrow 165^\circ$: stilstand
- $165^\circ \rightarrow 200^\circ$: $-10\,\mathrm{mm}$
- daarna stilstand tot $360^\circ$


In [ ]:
import numpy as np
import matplotlib.pyplot as plt


## Hulpfuncties

De afgeleiden worden omgerekend naar afgeleiden naar de nokhoek $\theta$ in radialen.


In [ ]:
def vijfde_graads(tau):
    """5de-graads bewegingswet uit les 5."""
    s = 6 * tau**5 - 15 * tau**4 + 10 * tau**3
    ds = 30 * tau**4 - 60 * tau**3 + 30 * tau**2
    d2s = 120 * tau**3 - 180 * tau**2 + 60 * tau
    d3s = 360 * tau**2 - 360 * tau + 60
    return s, ds, d2s, d3s


def cycloide(tau):
    """Cycloidale bewegingswet uit les 5."""
    s = tau - np.sin(2 * np.pi * tau) / (2 * np.pi)
    ds = 1 - np.cos(2 * np.pi * tau)
    d2s = 2 * np.pi * np.sin(2 * np.pi * tau)
    d3s = 4 * np.pi**2 * np.cos(2 * np.pi * tau)
    return s, ds, d2s, d3s


def voeg_stuk_toe(theta_deg, x, dx, ddx, dddx, start_deg, eind_deg, slag_mm, beginhoogte_mm, bewegingswet):
    beta = np.deg2rad(eind_deg - start_deg)
    mask = (theta_deg >= start_deg) & (theta_deg <= eind_deg)
    tau = (theta_deg[mask] - start_deg) / (eind_deg - start_deg)

    s, ds, d2s, d3s = bewegingswet(tau)
    x[mask] = beginhoogte_mm + slag_mm * s
    dx[mask] = slag_mm * ds / beta
    ddx[mask] = slag_mm * d2s / beta**2
    dddx[mask] = slag_mm * d3s / beta**3


def bouw_heffingswet(bewegingswet):
    theta_deg = np.linspace(0, 360, 3601)
    x = np.zeros_like(theta_deg)
    dx = np.zeros_like(theta_deg)
    ddx = np.zeros_like(theta_deg)
    dddx = np.zeros_like(theta_deg)

    # Stilstand 0-20 deg: x = 0 mm
    voeg_stuk_toe(theta_deg, x, dx, ddx, dddx, 20, 90, 25, 0, bewegingswet)

    # 90-155 deg: van 25 mm naar 10 mm
    x[theta_deg > 90] = 25
    voeg_stuk_toe(theta_deg, x, dx, ddx, dddx, 90, 155, -15, 25, bewegingswet)

    # Stilstand 155-165 deg: x = 10 mm
    x[theta_deg > 155] = 10

    # 165-200 deg: van 10 mm naar 0 mm
    voeg_stuk_toe(theta_deg, x, dx, ddx, dddx, 165, 200, -10, 10, bewegingswet)

    # Stilstand 200-360 deg: x = 0 mm
    x[theta_deg > 200] = 0
    return theta_deg, x, dx, ddx, dddx


def plot_heffingswet(theta_deg, x, dx, ddx, dddx, titel):
    fig, axs = plt.subplots(4, 1, figsize=(11, 10), sharex=True)

    grafieken = [
        (x, "Heffing x [mm]", "tab:blue"),
        (dx, "Snelheid dx/dtheta [mm/rad]", "tab:orange"),
        (ddx, "Versnelling d2x/dtheta2 [mm/rad2]", "tab:green"),
        (dddx, "Ruk d3x/dtheta3 [mm/rad3]", "tab:red"),
    ]

    for ax, (y, ylabel, color) in zip(axs, grafieken):
        ax.plot(theta_deg, y, color=color, linewidth=2)
        ax.set_ylabel(ylabel)
        ax.grid(True, alpha=0.3)
        ax.axhline(0, color="black", linewidth=0.8)
        for grens in [20, 90, 155, 165, 200]:
            ax.axvline(grens, color="gray", linestyle="--", linewidth=0.9, alpha=0.7)

    axs[-1].set_xlabel("Nokhoek theta [deg]")
    axs[0].set_title(titel)
    fig.tight_layout()
    plt.show()


# 1. 5de-graads bewegingswet

$$s(\tau)=6\tau^5-15\tau^4+10\tau^3$$


In [ ]:
theta_deg_5, x_5, dx_5, ddx_5, dddx_5 = bouw_heffingswet(vijfde_graads)
plot_heffingswet(theta_deg_5, x_5, dx_5, ddx_5, dddx_5, "Nokken systeem 28 - 5de-graads heffingswet")


# 2. Cycloidale bewegingswet

$$s(\tau)=\tau-\frac{\sin(2\pi\tau)}{2\pi}$$


In [ ]:
theta_deg_c, x_c, dx_c, ddx_c, dddx_c = bouw_heffingswet(cycloide)
plot_heffingswet(theta_deg_c, x_c, dx_c, ddx_c, dddx_c, "Nokken systeem 28 - cycloidale heffingswet")


# Vergelijking van de heffing


In [ ]:
plt.figure(figsize=(11, 4))
plt.plot(theta_deg_5, x_5, label="5de-graads", linewidth=2)
plt.plot(theta_deg_c, x_c, "--", label="cycloide", linewidth=2)
for grens in [20, 90, 155, 165, 200]:
    plt.axvline(grens, color="gray", linestyle="--", linewidth=0.9, alpha=0.7)
plt.xlabel("Nokhoek theta [deg]")
plt.ylabel("Heffing x [mm]")
plt.title("Vergelijking heffing: 5de-graads vs cycloide")
plt.grid(True, alpha=0.3)
plt.legend()
plt.tight_layout()
plt.show()


## Controle eindhoogtes


In [ ]:
print("theta = 0 deg   -> x = 0 mm")
print("theta = 90 deg  -> x = 25 mm")
print("theta = 155 deg -> x = 10 mm")
print("theta = 200 deg -> x = 0 mm")
